# 실기 대비
# 실전 문제풀이
# set 4

## 1) 데이터 및 시나리오

### 디지털프라자 매출 데이터

> 디지털프라자 A지점에서 최근 연휴 직전에 재고 처리와 매출 신장을 위해 대대적인 할인 행사를 하였다.  
> 하루만 반짝 진행한 행사에서 예상보다 많은 손님이 방문했고 이번에 발생한 매출데이터를 취합하여 향후 발송할 판촉물에 들어갈 컨텐츠를 기획하고자 한다.  
> 취합한 데이터는 다음과 같고 한 개의 행이 물품 1개 구매 내역이다.

### 데이터 개요

| 파일명 | 행 | 열 | 인코딩 |
|---|---:|---:|---|
| `sales_pos.csv` | 550068 | 11 | UTF-8 |

## 1) 데이터 및 시나리오

### 변수 상세

| 변수명 | 유형 | 설명 |
|---|---|---|
| `user` | int | 고객 식별자 |
| `prod` | string | 상품 식별자 |
| `gender` | string | 성별 |
| `age_group` | string | 연령대 |
| `job` | int | 직업 구분 |
| `city` | string | 도시 유형 구분 |
| `marital` | int | 결혼 여부 `(1: 결혼)` |
| `prod_cat1` | int | 상품 카테고리 `(1차)` |
| `prod_cat2` | int | 상품 카테고리 `(2차)` |
| `prod_cat3` | int | 상품 카테고리 `(3차)` |
| `purchase` | int | 결제 금액 |

## 2) 문제

### 필요 라이브러리 함수 및 클래스 목록

| 목록 |
|---|
| `from sklearn.preprocessing import MinMaxScaler` |
| `from sklearn.cluster import KMeans` |
| `from sklearn.metrics import silhouette_score` |

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('../../dataset/sales_pos.csv')

#df-sdis
display(df.shape)
display(df.dtypes)
display(df.isna().sum())

(550068, 11)

user           int64
prod          object
gender        object
age_group     object
job            int64
city          object
marital        int64
prod_cat1      int64
prod_cat2    float64
prod_cat3    float64
purchase       int64
dtype: object

user              0
prod              0
gender            0
age_group         0
job               0
city              0
marital           0
prod_cat1         0
prod_cat2    173638
prod_cat3    383247
purchase          0
dtype: int64

### Q01.

상품별 매출액(`purchase`)을 합산하여 그 매출액이 가장 큰 상품을 확인하고 해당 상품을 가장 많이 구매하는 직업(`job`)을 확인하시오.

※ 분석 결과를 기반으로 `job` 변수의 번호를 최종 출력하시오.  
※ 직업 확인시 상품 구매 개수를 기준으로 확인하시오. `(정답 예시: 1)`

In [22]:
df_q1 = df.copy()
#상품별 매출액
df_q1_gb = df_q1.groupby('prod')['purchase'].sum()
prod_max = df_q1_gb.idxmax()
display(type(df_q1_gb), df_q1_gb)
display(prod_max)

df_q1_prod_max = df_q1.loc[df_q1['prod'] == prod_max, :]
display(df_q1_prod_max)

df_q1_prod_max['job'].value_counts().idxmax()


pandas.core.series.Series

prod
P00000142    12837476
P00000242     3967496
P00000342     1296475
P00000442      441173
P00000542      807212
               ...   
P0099442      2870383
P0099642        83710
P0099742       991948
P0099842       737312
P0099942        78019
Name: purchase, Length: 3631, dtype: int64

'P00025442'

,user,prod,gender,age_group,job,city,marital,prod_cat1,prod_cat2,prod_cat3,purchase
667,130,P00025442,M,36-45,17,B,1,1,2.0,9.0,19706
749,142,P00025442,M,26-35,7,A,0,1,2.0,9.0,15212
833,150,P00025442,M,36-45,7,B,1,1,2.0,9.0,15255
1134,192,P00025442,M,18-25,1,B,0,1,2.0,9.0,15223
1205,198,P00025442,M,26-35,12,A,1,1,2.0,9.0,19296
...,...,...,...,...,...,...,...,...,...,...,...
544755,5854,P00025442,M,46-50,7,B,1,1,2.0,9.0,19072
544780,5858,P00025442,M,26-35,4,B,0,1,2.0,9.0,19665
545079,5911,P00025442,F,26-35,17,C,1,1,2.0,9.0,15284
545206,5933,P00025442,M,26-35,2,C,1,1,2.0,9.0,15338


4

### Q02.

결혼 여부(`marital`)에 따라 구매하는 물품의 종류가 많이 차이 나는지 확인하고자 한다.  
비교적 신혼부부가 많은 26-35세 그룹을 대상으로 각 고객의 구매물품 카테고리 개수를 산출하고 결혼여부별 그 평균값의 차이를 산출하시오.

※ 구매 물품의 카테고리 개수 산출에는 `prod_cat1`, `prod_cat2`, `prod_cat3` 변수를 사용한다.  
※ 카테고리 관련 변수의 결측치는 0으로 대체한다.  
※ 결측치를 대체한 데이터까지 포함하여 문제를 풀이하시오.  
※ 카테고리 관련 변수의 처리 예시는 다음과 같다.

| `prod_cat1` | `prod_cat2` | `prod_cat3` | `prod_cat` |
|---:|---:|---:|---|
| 1 | 2 | 0 | `1-2-0` |
| 8 | 0 | 0 | `8-0-0` |

※ 고객 식별자가 1인 고객은 총 21개 카테고리의 물품을 구매하였다.  
※ 정답은 절대값을 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [34]:
df_q2 = df.copy()
display(df_q2.isna().sum())
df_q2[['prod_cat2','prod_cat3']] = df_q2[['prod_cat2','prod_cat3']].fillna(0)
display(df_q2.isna().sum())
df_q2[['prod_cat1','prod_cat2','prod_cat3']] = df_q2[['prod_cat1','prod_cat2','prod_cat3']].astype(int).astype(str)
df_q2['prod_cat'] = df_q2['prod_cat1'] + '-' + df_q2['prod_cat2'] + '-' + df_q2['prod_cat3']
display(df_q2)

display(df_q2.loc[df_q2['user'] == 1, : ]['prod_cat'].nunique())

display(df_q2['age_group'].unique())
df_q2_2635 = df_q2.loc[df_q2['age_group'].isin(['26-35']), :]
df_q2_2635_prod_cat = df_q2_2635.groupby('user').agg(
    prod_cat=('prod_cat', 'nunique'),
    marital = ('marital', 'first')
)
display(df_q2_2635_prod_cat)
marital_0 = df_q2_2635_prod_cat.loc[df_q2_2635_prod_cat['marital'] == 0, :]['prod_cat'].mean()
marital_1 = df_q2_2635_prod_cat.loc[df_q2_2635_prod_cat['marital'] == 1, :]['prod_cat'].mean()

round(abs(marital_0 - marital_1), 2)

user              0
prod              0
gender            0
age_group         0
job               0
city              0
marital           0
prod_cat1         0
prod_cat2    173638
prod_cat3    383247
purchase          0
dtype: int64

user         0
prod         0
gender       0
age_group    0
job          0
city         0
marital      0
prod_cat1    0
prod_cat2    0
prod_cat3    0
purchase     0
dtype: int64

,user,prod,gender,age_group,job,city,marital,prod_cat1,prod_cat2,prod_cat3,purchase,prod_cat
0,1,P00069042,F,0-17,10,A,0,3,0,0,8370,3-0-0
1,1,P00248942,F,0-17,10,A,0,1,6,14,15200,1-6-14
2,1,P00087842,F,0-17,10,A,0,12,0,0,1422,12-0-0
3,1,P00085442,F,0-17,10,A,0,12,14,0,1057,12-14-0
4,2,P00285442,M,55+,16,C,0,8,0,0,7969,8-0-0
...,...,...,...,...,...,...,...,...,...,...,...,...
550063,6033,P00372445,M,51-55,13,B,1,20,0,0,368,20-0-0
550064,6035,P00375436,F,26-35,1,C,0,20,0,0,371,20-0-0
550065,6036,P00375436,F,26-35,15,B,1,20,0,0,137,20-0-0
550066,6038,P00375436,F,55+,1,C,0,20,0,0,365,20-0-0


21

array(['0-17', '55+', '26-35', '46-50', '51-55', '36-45', '18-25'],
      dtype=object)

,prod_cat,marital
user,,
3,18,0
5,43,1
8,32,1
9,31,0
11,34,0
...,...,...
6030,33,1
6034,8,0
6035,61,0


0.13

### Q03.

고객 5891명을 군집화 하여 각 군집별로 마케팅 전략을 수립하고자 한다.  
다음에 제시된 변수를 대상으로 k-means 군집분석을 실시하고 7개 군집으로 분석했을 때 Silhouette score를 산출하시오.

#### 독립변수

| 독립변수 |
|---|
| 성별 |
| 나이 |
| 직업 |
| 도시 |
| 결혼 여부 |
| 구매 상품 종류수 |
| 총 구매금액 |

※ 구매 상품 종류수 변수는 `prod` 변수를 참고하여 생성하시오.  
※ 성별 변수는 `gender` 변수에서 `M`을 1, `F`를 0으로 변환하여 사용하시오.  
※ 나이는 `age_group` 변수에서 나이가 가장 적은 그룹을 0으로 지정하고 가장 나이가 많은 그룹은 6으로 지정하는 방식으로 순서형 변수로 변환하시오.  
※ 직업과 도시 변수는 One Hot Encoding 변환하여 사용하시오.  
※ 군집 분석에 사용되는 변수는 총 29개이며 MinMax 정규화 후 분석하시오.  
※ seed는 `123`으로 지정하시오.  
※ 결과는 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [64]:
df_q3 = df.groupby('user').agg(
    gender=('gender', 'first'),
    age_group=('age_group','first'),
    job = ('job', 'first'),
    city = ('city', 'first'),
    marital = ('marital', 'first'),
    prod_num = ('prod', 'nunique'),
    total_purchase = ('purchase', 'sum')
)
display(df_q3.shape)
display(df_q3)
display(df_q3['gender'].value_counts())
df_q3['gender'] = df_q3['gender'].replace({"M": 1, "F": 0})
df_q3['gender'] = df_q3['gender'].astype(int)
display(df_q3['gender'].value_counts())

ser = pd.Series(df_q3['age_group'].unique(), name = 'age_group')
display(ser)
ser_sort =ser.sort_values().reset_index(drop = True)
display(ser_sort)
age_group_dict = dict(zip(ser_sort, ser_sort.index))
display(age_group_dict)
df_q3['age_group'] = df_q3['age_group'].replace(age_group_dict)
df_q3['age_group'] = df_q3['age_group'].astype(int)

df_q3_dummies = pd.get_dummies(df_q3, columns = ['job','city'])
display(df_q3_dummies.shape)

(5891, 7)

,gender,age_group,job,city,marital,prod_num,total_purchase
user,,,,,,,
1,F,0-17,10,A,0,35,334093
2,M,55+,16,C,0,77,810472
3,M,26-35,15,A,0,29,341635
4,M,46-50,7,B,1,14,206468
5,M,26-35,20,A,1,106,821001
...,...,...,...,...,...,...,...
6036,F,26-35,15,B,1,514,4116058
6037,F,46-50,1,C,0,122,1119538
6038,F,55+,1,C,0,12,90034


M    4225
F    1666
Name: gender, dtype: int64

1    4225
0    1666
Name: gender, dtype: int64

0     0-17
1      55+
2    26-35
3    46-50
4    51-55
5    36-45
6    18-25
Name: age_group, dtype: object

0     0-17
1    18-25
2    26-35
3    36-45
4    46-50
5    51-55
6      55+
Name: age_group, dtype: object

{'0-17': 0,
 '18-25': 1,
 '26-35': 2,
 '36-45': 3,
 '46-50': 4,
 '51-55': 5,
 '55+': 6}

(5891, 29)

In [63]:
#D
X = df_q3_dummies.copy()
display(X.dtypes)
#N
scaler = MinMaxScaler()
X_n = scaler.fit_transform(X)
#M
model = KMeans(n_clusters = 7,
               random_state = 123
               )
y_pred = model.fit_predict(X_n)
#E
sil = silhouette_score(X_n,y_pred)
round(sil, 2)

gender            int32
age_group         int32
marital           int64
prod_num          int64
total_purchase    int64
job_0             uint8
job_1             uint8
job_2             uint8
job_3             uint8
job_4             uint8
job_5             uint8
job_6             uint8
job_7             uint8
job_8             uint8
job_9             uint8
job_10            uint8
job_11            uint8
job_12            uint8
job_13            uint8
job_14            uint8
job_15            uint8
job_16            uint8
job_17            uint8
job_18            uint8
job_19            uint8
job_20            uint8
city_A            uint8
city_B            uint8
city_C            uint8
dtype: object

0.18